In [ ]:
# =============================================================================
# CELDA 1: IMPORTACIONES DEL PROYECTO Y LIBRERÍAS ESTÁNDAR
# =============================================================================
import sys
import os
import time
import json
import socket
import numpy as np
import matplotlib.pyplot as plt

# Aseguramos que Python encuentre la carpeta 'src' de surpRISe si ejecutamos desde 'notebooks/'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from surpRISe.components.antenna import Antenna
from surpRISe.components.ris import RIS
from surpRISe.channels.tx2ris2rx.tang_general_channel import TangGeneralChannel

print("✓ Módulos de surpRISe cargados correctamente.")

In [ ]:
# =============================================================================
# CELDA 2: PARÁMETROS DEL EXPERIMENTO (MODIFICABLES)
# =============================================================================

# --- 1. Conexión con ris_hardware_server.py (UDP) ---
SERVER_IP = "127.0.0.1"         # <--- [MODIFICABLE] IP del PC donde corre ris_hardware_server.py
UDP_PORT_RIS = 17000            # <--- [MODIFICABLE] Puerto UDP de la RIS (17000 por defecto)
ENVIAR_AL_SERVIDOR = True       # <--- [MODIFICABLE] True para mover la RIS física, False para solo simular
GUARDAR_CSV_CADA_PASO = False   # <--- [MODIFICABLE] False en tiempo real para no ralentizar el disco duro

# --- 2. Parámetros Físicos y de Frecuencia (FR2) ---
FREQ_HZ = 28e9                  # <--- [MODIFICABLE] Frecuencia de operación en Hz (28 GHz)
WAVELENGTH = 3e8 / FREQ_HZ      # Longitud de onda (~10.71 mm)
K0 = 2 * np.pi / WAVELENGTH     # Número de onda

# --- 3. Características de la Placa RIS (40x40 Dual-Pol) ---
RIS_NX = 40                     # <--- [MODIFICABLE] Columnas de la RIS (Eje X)
RIS_NY = 40                     # <--- [MODIFICABLE] Filas de la RIS (Eje Z/Y)
RIS_DX = WAVELENGTH / 2         # <--- [MODIFICABLE] Espaciado entre celdas en X (m)
RIS_DY = WAVELENGTH / 2         # <--- [MODIFICABLE] Espaciado entre celdas en Y (m)
RIS_BITS = 1                    # <--- [MODIFICABLE] 1 bit por polarización (estados 0 y 1)

# --- 4. Geometría Fija del Laboratorio (en metros respecto al centro de la RIS [0,0,0]) ---
RIS_POS = np.array([0.0, 0.0, 0.0])       # <--- [MODIFICABLE] Centro de la RIS
TX_POS  = np.array([0.0, 1.1, 0.0])       # <--- [MODIFICABLE] Posición fija de la bocina TX (m)

# --- 5. Parámetros de Potencia para Estimación Teórica en Vivo ---
TX_POWER_W = 1.0                          # <--- [MODIFICABLE] Potencia de transmisión de referencia (W)
NOISE_POWER_W = 1e-10                     # <--- [MODIFICABLE] Potencia de ruido (W)

print(f"✓ Configuración lista: RIS {RIS_NX}x{RIS_NY} ({RIS_NX*RIS_NY*2} bits totales) a {FREQ_HZ/1e9:.1f} GHz.")

In [ ]:
# =============================================================================
# CELDA 3: PRE-CÁLCULO ESTÁTICO (GEOMETRÍA RIS Y CANAL TX -> RIS)
# =============================================================================
# Como la RIS y la antena TX están fijas en el laboratorio, calculamos sus coordenadas
# y el canal incidente G una sola vez. Esto acelera el tracking en tiempo real x100.

def inicializar_escenario_estatico(tx_pos, ris_nx, ris_ny, dx, dy):
    """Pre-calcula la malla 3D de las 1600 celdas y el vector de canal TX -> RIS (G)."""
    N_total = ris_nx * ris_ny
    
    # Malla de coordenadas 3D de cada elemento de la RIS (plano XZ, normal en +Y)
    x_grid = (np.arange(ris_nx) - (ris_nx - 1) / 2.0) * dx
    z_grid = (np.arange(ris_ny) - (ris_ny - 1) / 2.0) * dy
    xx, zz = np.meshgrid(x_grid, z_grid)
    elem_coords = np.column_stack([xx.ravel(), np.zeros(N_total), zz.ravel()])

    # Canal estático TX -> RIS (Modelo General de Tang en Campo Cercano)
    vec_tx = tx_pos - elem_coords
    d_tx = np.linalg.norm(vec_tx, axis=1)
    cos_tx = np.clip(np.abs(vec_tx[:, 1]) / d_tx, 1e-6, 1.0)
    G_estatico = (np.sqrt(dx * dy * cos_tx) / (4 * np.pi * d_tx) * np.exp(-1j * K0 * d_tx))[:, None]
    
    return elem_coords, G_estatico

ELEM_COORDS, G_CANAL = inicializar_escenario_estatico(TX_POS, RIS_NX, RIS_NY, RIS_DX, RIS_DY)
print(f"✓ Escenario estático pre-calculado -> Celdas: {ELEM_COORDS.shape[0]}, Vector G: {G_CANAL.shape}")

In [ ]:
# =============================================================================
# CELDA 4: MOTOR DE OPTIMIZACIÓN RÁPIDA (1 USUARIO) Y ENVÍO UDP (3200 BITS)
# =============================================================================
# Mantiene intacta la función validada de 3200 bits y el envío JSON por UDP al puerto 17000.

# Creamos el socket UDP de forma persistente para no abrirlo y cerrarlo en cada milisegundo
sock_udp_global = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

def construir_payload_3200_bits(matriz_H_40x40: np.ndarray, matriz_V_40x40: np.ndarray = None) -> tuple[np.ndarray, bytes]:
    """
    Construye la trama de configuración de 3200 bits (400 bytes) para la RIS 40x40.
    Mantiene exactamente el orden Row-Major ('C'): 1600 bits H seguidos de 1600 bits V.
    """
    assert matriz_H_40x40.shape == (40, 40), "La matriz H debe ser de 40x40"
    
    if matriz_V_40x40 is None:
        matriz_V_40x40 = matriz_H_40x40.copy()
    else:
        assert matriz_V_40x40.shape == (40, 40), "La matriz V debe ser de 40x40"
        
    bits_H = matriz_H_40x40.astype(np.uint8).flatten(order='C')  # [0..1599]
    bits_V = matriz_V_40x40.astype(np.uint8).flatten(order='C')  # [1600..3199]
    
    vector_bits_3200 = np.concatenate([bits_H, bits_V])
    payload_bytes_400 = np.packbits(vector_bits_3200, bitorder='big').tobytes()
    
    return vector_bits_3200, payload_bytes_400


def enviar_configuracion_a_ris_rt(matriz_estados_40x40, nombre_paso="rt"):
    """Envía la matriz 40x40 al servidor UDP de la RIS con mínima latencia."""
    bits_3200, bytes_400 = construir_payload_3200_bits(matriz_estados_40x40)

    if GUARDAR_CSV_CADA_PASO:
        np.savetxt(f"config_ris_{nombre_paso}.csv", matriz_estados_40x40, fmt="%d", delimiter=",")

    if not ENVIAR_AL_SERVIDOR:
        return bits_3200

    # Estructura exacta validada en tu test_wmmse_experimento_completo.py y ris_hardware_server.py
    mensaje_json = {
        "command": "RIS_SET_CONF",
        "payload": bits_3200.tolist()
    }
    datos_para_enviar = json.dumps(mensaje_json).encode("utf-8")

    try:
        sock_udp_global.sendto(datos_para_enviar, (SERVER_IP, UDP_PORT_RIS))
    except Exception as e:
        print(f"   ⚠ [AVISO UDP]: Error enviando a {SERVER_IP}:{UDP_PORT_RIS} ({e})")
        
    return bits_3200


def actualizar_ris_para_posicion_rx(rx_pos: np.ndarray, nombre_paso="tracking"):
    """
    Dada la nueva coordenada 3D [x, y, z] del usuario:
    1. Calcula el canal rápido RIS -> RX (Tang General).
    2. Aplica alineación analítica cerrada (SCRPA / WMMSE K=1).
    3. Cuantiza a 1 bit (40x40) y envía los 3200 bits por UDP a la RIS.
    """
    t0 = time.perf_counter()
    
    # 1. Canal dinámico RIS -> Usuario (h_r) en campo cercano
    vec_rx = rx_pos - ELEM_COORDS
    d_rx = np.linalg.norm(vec_rx, axis=1)
    cos_rx = np.clip(np.abs(vec_rx[:, 1]) / d_rx, 1e-6, 1.0)
    h_r = (np.sqrt(RIS_DX * RIS_DY * cos_rx) / (4 * np.pi * d_rx)) * np.exp(-1j * K0 * d_rx)

    # 2. Canal en cascada: a = h_r^* * g
    a_cascada = h_r.conj() * G_CANAL[:, 0]

    # 3. Fase óptima continua (SCRPA para Usuario Único) y cuantización
    fases_opt_cont = np.mod(-np.angle(a_cascada), 2 * np.pi)
    niveles = 2 ** RIS_BITS
    paso_rad = (2 * np.pi) / niveles
    estados_int = (np.round(fases_opt_cont / paso_rad).astype(int) % niveles).reshape((RIS_NX, RIS_NY))
    fases_q_rad = estados_int.ravel() * paso_rad

    # 4. Envío inmediato por UDP al servidor de la RIS
    enviar_configuracion_a_ris_rt(estados_int, nombre_paso=nombre_paso)
    
    t_ms = (time.perf_counter() - t0) * 1000.0

    # 5. Métricas teóricas de control (Potencia recibida y Capacidad)
    theta_q = np.exp(1j * fases_q_rad)
    h_eff = np.vdot(a_cascada, theta_q)
    p_rx_w = TX_POWER_W * (np.abs(h_eff) ** 2)
    p_rx_dbm = 10 * np.log10(max(p_rx_w, 1e-20) / 1e-3)
    rate_bps_hz = float(np.log2(1.0 + p_rx_w / NOISE_POWER_W))

    return {
        "rx_pos": rx_pos.copy(),
        "estados_40x40": estados_int,
        "fases_deg": np.rad2deg(fases_q_rad).reshape((RIS_NX, RIS_NY)),
        "p_rx_dbm": p_rx_dbm,
        "rate_bps_hz": rate_bps_hz,
        "latencia_ms": t_ms
    }

print("✓ Motor de seguimiento en tiempo real inicializado.")

In [ ]:
# =============================================================================
# CELDA 5: MODO A - SEGUIMIENTO AUTOMÁTICO DE TRAYECTORIA EN TIEMPO REAL
# =============================================================================
# Define un punto inicial, un punto final y el número de pasos.
# El script moverá el foco de la RIS en tiempo real siguiendo la trayectoria.

POS_INICIAL = np.array([-0.6, 1.4, 0.0])  # <--- [MODIFICABLE] Coordenada inicial del usuario [x, y, z] (m)
POS_FINAL   = np.array([ 0.6, 1.4, 0.0])  # <--- [MODIFICABLE] Coordenada final del usuario [x, y, z] (m)
NUM_PASOS   = 25                          # <--- [MODIFICABLE] Número de puntos en la trayectoria
PAUSA_ENTRE_PASOS_SEG = 0.5               # <--- [MODIFICABLE] Segundos entre cada actualización (ej. 0.5 s)
PAUSA_MANUAL_VNA = False                  # <--- [MODIFICABLE] Pon True si quieres pulsar [ENTER] en cada paso

# Generamos los puntos 3D de la trayectoria
trayectoria_xyz = np.linspace(POS_INICIAL, POS_FINAL, NUM_PASOS)
historico_tracking = []

print("=========================================================================")
print(f" INICIANDO SEGUIMIENTO EN TIEMPO REAL ({NUM_PASOS} posiciones)")
print("=========================================================================")

try:
    for idx, pos_actual in enumerate(trayectoria_xyz):
        info = actualizar_ris_para_posicion_rx(pos_actual, nombre_paso=f"track_{idx:03d}")
        historico_tracking.append(info)
        
        x, y, z = pos_actual
        print(
            f"[{idx+1:02d}/{NUM_PASOS:02d}] Pos RX: ({x:+.2f}, {y:.2f}, {z:+.2f}) m | "
            f"Prx est: {info['p_rx_dbm']:6.2f} dBm | "
            f"Cálculo+UDP: {info['latencia_ms']:.2f} ms"
        )
        
        if PAUSA_MANUAL_VNA:
            input("   >>> Realiza tu medida en el VNA y pulsa [ENTER] para el siguiente punto...")
        else:
            time.sleep(PAUSA_ENTRE_PASOS_SEG)

    print("\n✓ Seguimiento de trayectoria completado con éxito.")
except KeyboardInterrupt:
    print("\n⏹ Seguimiento detenido manualmente por el usuario.")

In [ ]:
# =============================================================================
# CELDA 6: MODO B - SEGUIMIENTO INTERACTIVO MANUAL (COORDENADAS POR TECLADO)
# =============================================================================
# Ejecuta esta celda si estás moviendo el soporte de la antena en el laboratorio
# y quieres escribir tú mismo la nueva coordenada "x y z" para que la RIS apunte al instante.

print("=========================================================================")
print(" MODO INTERACTIVO: Escribe las coordenadas 'x y z' en metros (ej: -0.3 1.4 0.0)")
print(" Escribe 'q' o deja vacío y pulsa [ENTER] para salir.")
print("=========================================================================")

while True:
    entrada = input("\nNueva posición RX [x y z] (m) >>> ").strip()
    if not entrada or entrada.lower() == "q":
        print("Saliendo del modo interactivo.")
        break
    try:
        partes = [float(val) for val in entrada.replace(",", " ").split()]
        if len(partes) != 3:
            print("⚠ Formato incorrecto. Introduce exactamente 3 valores separados por espacio (ej: 0.2 1.5 0.0).")
            continue
        nueva_pos = np.array(partes)
        info = actualizar_ris_para_posicion_rx(nueva_pos, nombre_paso="interactivo")
        print(
            f"   ✓ RIS actualizada en {info['latencia_ms']:.2f} ms -> Apuntando a {nueva_pos} m | "
            f"Prx teórica: {info['p_rx_dbm']:.2f} dBm"
        )
    except ValueError:
        print("⚠ Error: Introduce números válidos.")

In [ ]:
# =============================================================================
# CELDA 7: MODO C - SEGUIMIENTO EN VIVO VÍA UDP EXTERNO (ESTILO app2.py)
# =============================================================================
# Escucha coordenadas [x, y, z] por UDP en el puerto 5005 (como hace ris-fr2-gui/app2.py)
# desde un sistema de tracking externo (cámara, UWB, script emisor) y reconfigura la RIS al vuelo.

UDP_TRACKER_LISTEN_IP = "0.0.0.0"   # <--- [MODIFICABLE] Escuchar en todas las interfaces
UDP_TRACKER_LISTEN_PORT = 5005      # <--- [MODIFICABLE] Puerto donde tu sensor externo envía {"x":.., "y":.., "z":..}
UMBRAL_MOVIMIENTO_M = 0.01          # <--- [MODIFICABLE] Solo reconfigura la RIS si el usuario se mueve > 1 cm

def escuchar_sensor_externo_udp():
    sock_rx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock_rx.bind((UDP_TRACKER_LISTEN_IP, UDP_TRACKER_LISTEN_PORT))
    sock_rx.settimeout(1.0)
    
    ultima_pos = None
    print(f"📡 Escuchando coordenadas de usuario en UDP {UDP_TRACKER_LISTEN_IP}:{UDP_TRACKER_LISTEN_PORT}...")
    print("   Formatos aceptados: JSON '{\"x\": -0.2, \"y\": 1.4, \"z\": 0.0}' o texto '-0.2,1.4,0.0'")
    print("   Pulsa el botón 'Stop' de la celda (KeyboardInterrupt) para detener.")

    try:
        while True:
            try:
                data, _ = sock_rx.recvfrom(1024)
                texto = data.decode("utf-8", errors="ignore").strip()
                
                # Soportamos tanto JSON como valores separados por coma
                if texto.startswith("{"):
                    obj = json.loads(texto)
                    pos_rx = np.array([float(obj["x"]), float(obj["y"]), float(obj["z"])])
                else:
                    partes = [float(v) for v in texto.replace(",", " ").split()]
                    pos_rx = np.array(partes[:3])

                # Filtramos fluctuaciones menores a UMBRAL_MOVIMIENTO_M
                if ultima_pos is None or np.linalg.norm(pos_rx - ultima_pos) >= UMBRAL_MOVIMIENTO_M:
                    info = actualizar_ris_para_posicion_rx(pos_rx, nombre_paso="sensor_udp")
                    ultima_pos = pos_rx
                    print(
                        f"🎯 [TRACKING VIVO] RX=({pos_rx[0]:+.2f}, {pos_rx[1]:.2f}, {pos_rx[2]:+.2f}) m | "
                        f"Prx={info['p_rx_dbm']:6.2f} dBm | Latencia={info['latencia_ms']:.2f} ms"
                    )
            except socket.timeout:
                continue
    except KeyboardInterrupt:
        print("\n⏹ Recepción de sensor UDP detenida.")
    finally:
        sock_rx.close()

# Descomenta la siguiente línea cuando quieras activar la escucha en vivo por UDP:
# escuchar_sensor_externo_udp()

In [ ]:
# =============================================================================
# CELDA 8: VISUALIZACIÓN DE LA TRAYECTORIA Y EVOLUCIÓN DE LOS MAPAS DE LA RIS
# =============================================================================
# Muestra cómo se desplazó el anillo de enfoque de Fresnel (campo cercano) sobre
# la placa de 40x40 elementos durante la trayectoria ejecutada en la Celda 5.

if len(historico_tracking) >= 3:
    indices_muestra = [0, len(historico_tracking) // 2, len(historico_tracking) - 1]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, idx_m in zip(axes, indices_muestra):
        item = historico_tracking[idx_m]
        px, py, pz = item["rx_pos"]
        im = ax.imshow(item["estados_40x40"], cmap="binary", vmin=0, vmax=1)
        ax.set_title(f"Paso {idx_m+1}: RX=({px:+.2f}, {py:.2f}, {pz:+.2f})m")
        ax.set_xlabel("Columna X (0..39)")
        ax.set_ylabel("Fila Y (0..39)")

    plt.suptitle("Evolución Espacial del Patrón Binario 40x40 durante el Seguimiento", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Gráfica de la trayectoria y potencia teórica
    xs = [it["rx_pos"][0] for it in historico_tracking]
    prx = [it["p_rx_dbm"] for it in historico_tracking]

    plt.figure(figsize=(7, 3.5))
    plt.plot(xs, prx, marker="o", markersize=4)
    plt.title("Potencia Teórica Recibida a lo largo del Recorrido (Single-User Tracking)")
    plt.xlabel("Posición X del Usuario (m)")
    plt.ylabel("Potencia Recibida Estimada (dBm)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("Ejecuta primero la Celda 5 con al menos 3 pasos para visualizar la evolución.")